Transformers 라이브러리

인코더 기반 모델: 예시 모델 Multiple Choice
- 문제 예시) 여러개 입력이 주어졌을때 주어진 문장중 옳은 문장을 고르는 객관식 문제, 
- 입력 프롬프트 한개와 그에 대응하는 n개 후보로 구성됨
- 프롬프트 + 답변1, 프롬프트 + 답변2,...프롬프트 + 답변n 목록을 모두 입력 받아 후보중 무엇이 정답인지 출력

1. klue/roberta-large 기본 모델: BertForMultipleChoice
- 추론 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)

2. klue/roberta-large 기본 모델: BertForMultipleChoice
- 파인튜닝 : 수능 국어 문제 풀이 - 5지선다형(보기 5개)
- 학습용 데이터셋 준비: 데이터셋(수능 문제 풀이 HAERAE-HUB/csatqa)
- 데이터셋 전처리(토크나이징), 샘플링
- 다중선택(MultipleChoice) 모데을 위한 동적 패딩 전용 커스텀 데이터 콜레이터(Data Collator)
- 트레이너(Trainer) 설정 
- 학습 실행
- 모델 저장
- 기본 VRAM 환원 패턴 (모델·옵티마이저·텐서 해제)
- 모델 로드 및 모델 추론

In [1]:
# 1. 환경/디바이스 설정 및 사전학습 모델 로드
import os # 시스템
import re # 정규식
from dataclasses import dataclass # @dataclass 붙이면 일반 클래스에서 반복 코드를 자동 생성(생성자 변수..)
from typing import Any, Optional, Union # 매개변수 속성 지정

import torch
from datasets import load_dataset
from transformers import(
    AutoModelForMultipleChoice,
    AutoTokenizer,
    EarlyStoppingCallback,
    PreTrainedTokenizerBase, # datacollator 에서 토크나이저임을 명시 할때 사용
    Trainer,
    TrainingArguments,
)
from transformers.utils import PaddingStrategy

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if torch.cuda.is_available():
    total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({total_vram_gb:.2f} GB)")

# 모델 이름 지정
model_name = "klue/roberta-large"

# 사전학습 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMultipleChoice.from_pretrained(model_name)

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU (6.00 GB)


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForMultipleChoice LOAD REPORT from: klue/roberta-large
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | MISSING    | 
roberta.pooler.dense.bias   | MISSING    | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
# 2. 데이터셋 로드: HAERAE-HUB/csatqa
dataset = load_dataset(
    path="HAERAE-HUB/csatqa",
    name="full"
)
print("Splits:", list(dataset.keys()))

# 데이터셋 개수 및 데이터셋 컬럼
for split_name in dataset.keys():
    print(f" - {split_name}: {len(dataset[split_name])} rows")
    print(f" - cols={dataset[split_name].column_names}")

# 데이터셋 구조
full_data = dataset["test"]
sample_0 = full_data[0]
print(f"\ntest[0] 구조 확인: {sample_0}\n")
for k, v in sample_0.items():
    preview = str(v)[:140].replace("\n", " ")
    print(f"{k} ({type(v).__name__}): {preview}")

# gold 값 분포 확인 (0~4 기반인지, 1~5 기반인지 확인 필요)
print(f"\ngold 값 고유값 샘플: {sorted(set(full_data["gold"]))}")

# 데이터 분할: train:validation 8:2
split_dataset = full_data.train_test_split(test_size=0.2, seed=42)
print(f"\n분할 후 크기: {split_dataset}")

Splits: ['test']
 - test: 936 rows
 - cols=['question', 'context', 'option#1', 'option#2', 'option#3', 'option#4', 'option#5', 'gold', 'category', 'human_performance']

test[0] 구조 확인: {'question': ' 이 이야기에서 얻을 수 있는 교훈으로 가장 적절한 것은?', 'context': '이제 한 편의 이야기를 들려 드립니다. 잘 듣고 물음에 답하십시오.\n자, 여러분! 안녕하십니까? 오늘은 제가 어제 꾼 꿈 이야기 하날 들려 드리겠습니다. 전 꿈속에서 낯선 거리를 걷고 있었습니다. 그러다가 홍미로운 간판을 발견했답니다. 행 복을 파는 가게. 그렇게 쓰여 있었습니다. 전 호기심으로 문을 열고 들어갔답니다. 그곳 에서는 한 노인이 물건을 팔고 있었습니다. 전 잠시 머뭇거리다가 노인에게 다가가서 물 었습니다. 여기서는 무슨 물건을 파느냐고요. 노인은 미소를 지으며, 원하는 것은 뭐든 다 살 수 있다고 말했습니다. 저는 제 귀를 의심했습니다. \'무엇이든 다?\' 전 무엇을 사야 할까 생각하다가 말했답니다. "사랑, 부귀 그리고 지혜하고 건강도 사고 싶습니다. 저 자신뿐 아니라 우리 가족 모두 를 위해서요. 지금 바로 살 수 있나요?" 그러자 노인은 빙긋이 웃으며 대답했습니다. "젊은이, 한번 잘 보게나. 여기에서 팔고 있는 것은 무르익은 과일이 아니라 씨앗이라 네. 앞으로 좋은 열매를 맺으려면 이 씨앗들을 잘 가꾸어야 할 걸세."', 'option#1': '새로운 세계에 대한 열망을 가져야 한다.', 'option#2': '주어진 기회를 능동적으로 활용해야 한다.', 'option#3': '큰 것을 얻으려면 작은 것은 버려야 한다.', 'option#4': '물질적 가치보다 정신적 가치를 중시해야 한다.', 'option#5': '소망하는 바를 성취하기 위해서는 노력을 해야 한다.', 'gold': 5, 'catego

In [ ]:
# 3. 텍스트 정제 및 구조 변환
# - 지문(context) 질문(question) 선택지(option#1~#5)에 섞여 있는 
# - 불필요한 LaTex 명령어(\text{...} 등), 연속 공백 줄바꿈을 정리한다
import re
from typing import Any, Optional

# 텍스트 정제 함수
def clean_csat_text(text: Optional[str]) -> str:
    if not text:
        return ""

    # 불필요한 LaTex 명령어(\text{...} 등), 연속 공백 줄바꿈 정리
    # \text{내용} 형태에서 괄호 안 내용만 추출 (그룹 1 사용)
    text = re.sub(r"\\text\{([^}]*)\}", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 텍스트 구조 함수
def format_example(example: dict[str, Any]) -> dict[str, Any]:
    # question, context, option#1~#5, gold 값 추출
    question = clean_csat_text(example.get("question"))
    context = clean_csat_text(example.get("context"))
    options = [
        clean_csat_text(example.get(f"option#{i}")) for i in range(1, 6)
    ]
    label = int(example["gold"]) - 1 # 1~5 -> 0~4 변환

    return {
        "question": question,
        "context": context,
        "options": options,
        "label": label,
    }

# 텍스트 구조 및 정제 함수 호출
cleaned_dataset = split_dataset.map(function=format_example)

# 변환 결과 검증
print("컬럼 목록:", cleaned_dataset["train"].column_names)
print("첫 샘플 label (0~4):", cleaned_dataset["train"][0]["label"])
print("첫 샘플 options 개수:", len(cleaned_dataset["train"][0]["options"]))
print("첫 샘플 options 미리보기:", cleaned_dataset["train"][0]["options"])

컬럼 목록: ['question', 'context', 'option#1', 'option#2', 'option#3', 'option#4', 'option#5', 'gold', 'category', 'human_performance', 'options', 'label']
첫 샘플 label (0~4): 1
첫 샘플 options 개수: 5
첫 샘플 options 미리보기: ['[A]는 [B]와 달리 대조를 통해 주제 의식을 강조한다.', '[A]는 [B]와 달리 유사한 구절을 병치하여 운율감을 조성한다.', '[B]는 [A]와 달리 공감각적 심상을 통해 입체감을 부여한다.', '[B]는 [A]와 달리 현재 시제를 사용하여 현장감을 부각한다.', '[B]는 [A]와 달리 의성어를 통해 구체적인 생동감을 부여한다.']
